# Week 6: Attention

## Text as Data

Professor: Elliott Ash, NYU

TA: Eduardo Zago, NYU

Where are we now?

At this point, you know how to prepare the input text for training LLMs by splitting text into individual word and subword tokens, which can be encoded into vector representations, embeddings, for the LLM.

Objectives:

1) Data sampling with a sliding window
2) Token embeddings + positional embeddings
3) Attention
4) Attention
5) Attention

Code adapted from Raschka Ch: 2.6 , 2.7, 3.1-3.3

In [1]:
# set random seed
!pip install torch --upgrade
!pip install gensim
!pip install tiktoken

import collections
from collections import Counter

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch.optim as optim
from tqdm.notebook import tqdm
import re
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from string import punctuation
nltk.download('stopwords')
from nltk.corpus import stopwords
stoplist = set(stopwords.words('english'))

from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/joshuaberry/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/joshuaberry/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
# If you are using Google Colab, here's the code to load sc_cases_cleaned.pkl from local.
from google.colab import files
uploaded = files.upload()

Saving sc_cases_cleaned.pkl to sc_cases_cleaned.pkl


In [2]:
df = pd.read_pickle('sc_cases_cleaned.pkl',compression='gzip')

df = df.reset_index(drop = True)
df = df.assign(authorship_id=(df['authorship']).astype('category').cat.codes) # Authorship as categorical
df['index'] = df['authorship'] + df['date_standard'].astype(str)
df.rename(columns = {'x_republican':'republican'}, inplace = True)

# Basic preprocessing for the dataset
translator = str.maketrans(' ', ' ', punctuation)
from nltk.tokenize import word_tokenize

def preprocess(doc):
  doc = doc.replace('\r', ' ').replace('\n', ' ')
  doc = re.sub(r"(\d)([A-Za-z])", r"\1 \2", doc) # separate numbers from strings
  doc = re.sub(r"([A-Za-z])(\d)", r"\1 \2", doc) # separate strings from numbers
  d = doc.translate(translator).lower() # remove punctuation
  words = word_tokenize(d)
  words = [w for w in words if w not in stoplist] # remove stopwords
  words = [w if not w.isdigit() else '#' for w in words] # normalize numbers
  output = ' '.join(words) # Let's not tokenize now
  return output

df['opinion_text_clean'] = df['opinion_text'].apply(preprocess)

Supervised embeddings are limited because they learn semantic relationships only insofar as they help predict a specific outcome. As a result, word meanings are shaped by the task rather than by broader linguistic structure. In contrast, the inherent structure of language provides its own training signal, allowing deep neural networks to learn general semantic patterns through self-supervised objectives.

1) Next sentence prediction

2) Masked word prediction

These tasks allow models to learn general semantic and syntactic relationships without relying on externally labeled data. We have to learn how to build input target pairs. This is called data sampling with a sliding window.

In [3]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

comp = df['opinion_text_clean'][0]
vocab = tokenizer.encode(comp, allowed_special={"<|endoftext|>"})
integers = vocab[50:60]
words = tokenizer.decode(integers)

print(integers)
print(words)

[2717, 12580, 2753, 23993, 1277, 5198, 8492, 11375, 1306, 8853]
 federal prosecution takes unsuccessful direct appeal judgment conviction next petition


In [4]:
# Easiest and most intuitive way:

context_size = 4         #1
x = integers[:context_size]
y = integers[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [2717, 12580, 2753, 23993]
y:      [12580, 2753, 23993, 1277]


In [5]:
for i in range(1, context_size+1):
    context = integers[:i]
    desired = integers[i]
    print(context, "---->", desired)

[2717] ----> 12580
[2717, 12580] ----> 2753
[2717, 12580, 2753] ----> 23993
[2717, 12580, 2753, 23993] ----> 1277


In [6]:
for i in range(1, context_size+1):
    context = integers[:i]
    desired = integers[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 federal ---->  prosecution
 federal prosecution ---->  takes
 federal prosecution takes ---->  unsuccessful
 federal prosecution takes unsuccessful ---->  direct


Now: we need to build a PyTorch DataLoader that generates an input tensor containing the text that the LLM sees and a target tensor that includes the targets for the LLM to predict, to operationalize the predictive capabilities of the model/embeddings.


In [7]:
from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt) # Tokenizes the entire text

        for i in range(0, len(token_ids) - max_length, stride):  # Uses a sliding window to chunk the book into overlapping sequences of max_length
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self): # Returns the total number of rows in the dataset
        return len(self.input_ids)

    def __getitem__(self, idx):  # Returns a single row from the dataset
        return self.input_ids[idx], self.target_ids[idx]

# Now A data loader to generate batches with input-with pairs
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2") # Initializes the tokenizer
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # Creates dataset
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training. (See Rashkca A6)
        num_workers=num_workers  # The number of CPU processes to use for preprocessing
    )

    return dataloader

In [8]:
dataloader = create_dataloader_v1(comp, batch_size=1,
                                  max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)      #1
first_batch = next(data_iter)
print(first_batch) # Input and target tensors

[tensor([[31012,   308,  1040,  7423]]), tensor([[ 308, 1040, 7423, 6793]])]


For homework I will ask you to play with the stride and max lengths for you to understand this better.



Now, last lab we saw how to build embedding layers using PyTorch. These were static CBOW representations of each sequence (remember that we took the mean). Now we would also like to add positional context to these embeddings. For didactic purpose let's keep working with the small example

In [9]:
vocab_size = 10
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035],
        [-0.5880,  0.3486,  0.6603],
        [-0.2196, -0.3792,  0.7671],
        [-1.1925,  0.6984, -1.4097],
        [ 0.1794,  1.8951,  1.3689],
        [-1.6033, -1.3250,  0.1784],
        [-2.1338,  1.0524, -0.3885],
        [-0.9343,  1.8319, -0.3378],
        [ 0.8805,  1.5542,  0.6266],
        [-0.1755,  0.0983, -0.0935]], requires_grad=True)


In [10]:
vocab_size = 50257 # Size of gpt-2 vocab
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    comp, batch_size=8, max_length=max_length,
   stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[31012,   308,  1040,  7423],
        [ 6793,  4459,  2184,  6268],
        [ 2717, 17234,  1281, 42946],
        [ 2867,  8259,  1303,   514],
        [   66,  8460,  1303,  2426],
        [  530,  1941,   640, 17385],
        [ 4143,  4539,  3128,  8492],
        [11375,  4329,  2457,  8460]])

Inputs shape:
 torch.Size([8, 4])


In [11]:
# Define standard token embedding
token_embeddings = token_embedding_layer(inputs)

# Define positional embedding layer
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)


torch.Size([4, 256])


In [12]:
print(pos_embeddings)

tensor([[-0.6424, -0.1346,  0.0174,  ..., -0.6637,  1.1505, -0.7226],
        [ 0.0157, -0.1880,  0.2333,  ...,  0.6377,  0.2461,  0.3518],
        [ 0.1129,  1.2428,  0.3310,  ...,  0.4713,  0.4612,  0.9181],
        [-0.2098, -0.5736,  1.0646,  ..., -0.4725,  0.9837, -0.7215]],
       grad_fn=<EmbeddingBackward0>)


In [13]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape) # 4 input words, 8 different texts (batch_size, 256 elements per vector)

torch.Size([8, 4, 256])


Now that we have basically all the pre-processing, let's implement the processing of these text for prediction. The algorithm is called Attention:

We will see these elements in isolation, as is important to understand them by themselves.

For now, we will see a attention framework without trainable weights.

In [14]:
# Keep the first sentence:

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

### Attention scores:

Dot products $x_1^{T}x_2$

Recall:

$$\cos θ = \frac{\mathbf{x}^{\top}\mathbf{y}}{\|\mathbf{x}\|\|\mathbf{y}\|}$$

We are basically calculating the similarity that the embeddings have with respect to input 1 (the query).

In [15]:
query = inputs[1] #
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### Attention weights:

Raschka: "The main goal behind the normalization is to obtain attention weights that sum up to 1. This normalization is a convention that is useful for interpretation and maintaining training stability in an LLM."

In [16]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


Better to do it with a SoftMax:

In [17]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [18]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


### Context Vectors $z(.)$

Finally, we obtain the context vectors by multiplying each input vector by their relative attention weight: effectively gives more weight to embeddings more similar to the query.

In [19]:
query = inputs[1]         #1
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


All together with Torch:

In [20]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
all_context_vecs = attn_weights @ inputs

print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [21]:
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

### Fine tuning BERT

Finally, let's see how we can already use the transformer arquitecture for our own classification tasks:

In [22]:
# ── 1. Load & split data ───────────────────────────────────────────────────────
df = df[['opinion_text_clean', 'republican']].dropna()
df['republican'] = df['republican'].astype(int)  # ensure 0/1

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42,
                                     stratify=df['republican'])

# ── 2. Dataset ─────────────────────────────────────────────────────────────────
class OpinionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── 3. Tokenizer & dataloaders ─────────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = OpinionDataset(train_df['opinion_text_clean'], train_df['republican'], tokenizer)
val_dataset   = OpinionDataset(val_df['opinion_text_clean'],   val_df['republican'],   tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)


In [25]:
# ── 4. Model ───────────────────────────────────────────────────────────────────
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)
model.to(device)

# ── 5. Optimizer & scheduler ───────────────────────────────────────────────────
EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# ── 6. Training loop ───────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in tqdm(loader, desc="Training"):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent exploding gradients
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

def eval_epoch(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
# ── 7. Run training ────────────────────────────────────────────────────────────
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, device)

    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

# ── 8. Final report & save ─────────────────────────────────────────────────────
print("\nClassification Report:")
print(classification_report(val_labels, val_preds, target_names=['Democrat', 'Republican']))



Epoch 1/3


Evaluating: 100%|██████████| 20/20 [00:29<00:00,  1.47s/it]


  Train Loss: 0.5501 | Train Acc: 0.7736
  Val   Loss: 0.5447 | Val   Acc: 0.7727

Epoch 2/3


Evaluating: 100%|██████████| 20/20 [00:20<00:00,  1.04s/it]


  Train Loss: 0.5323 | Train Acc: 0.7736
  Val   Loss: 0.5242 | Val   Acc: 0.7727

Epoch 3/3


Evaluating: 100%|██████████| 20/20 [00:21<00:00,  1.06s/it]
/opt/anaconda3/envs/tad_courses/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/tad_courses/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/tad_courses/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

  Train Loss: 0.4751 | Train Acc: 0.7687
  Val   Loss: 0.5097 | Val   Acc: 0.7727

Classification Report:
              precision    recall  f1-score   support

    Democrat       0.00      0.00      0.00        35
  Republican       0.77      1.00      0.87       119

    accuracy                           0.77       154
   macro avg       0.39      0.50      0.44       154
weighted avg       0.60      0.77      0.67       154

